In [ ]:
import requests
import json
import glob

# You need a file config.py with a variable called api_key containing your key
from config import api_key

### Configuration

In [ ]:
batch_size = 100  # prompt size (e.g. number of lines), adjust if needed
system_prompt = "Translate the input text to German. Return only the translations, in the same format as the input, don't leave any comments"
model = "gpt-5.1"

url = "https://llmproxy.uva.nl/chat/completions"


In [ ]:
headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "Authorization": f"Bearer {api_key}"
}

### Process transcripts

In [ ]:
# Read all .txt in folder /transcripts
docs = []
for file in glob.glob("transcripts/*.txt"):
    with open(file, "r") as f:
        lines = f.read().splitlines()
    docs.append(lines)

### Model requests

In [ ]:
responses = []
for doc in docs:
    for i in range(0, len(doc), batch_size):
        lines = doc[i:i+batch_size]
        data = {
            "model": model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": "\n".join(lines)}
            ]
        }
        success = False
        while not success:
            r = requests.post(url=url, data=json.dumps(data), headers=headers)
            if r.status_code == 200:
                success = True
        responses.append(r)

translations = [r.json()["choices"][0]["message"]["content"] for r in responses]
translations = "\n".join(translations)
with open("transcripts/translations.txt", "w") as f:
    f.write(translations)
